In [ ]:
# CELL 1 — Imports & Style Setup
# Loads libraries and applies a dark FinTech-style chart theme
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import calendar
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0E0E16', 'axes.facecolor':   '#14141F',
    'axes.edgecolor':   '#1E1E2E', 'axes.labelcolor':  '#CDD6F4',
    'xtick.color':      '#6C7086', 'ytick.color':      '#6C7086',
    'grid.color':       '#1E1E2E', 'grid.alpha':        0.7,
    'text.color':       '#E6E8EF', 'font.family':      'DejaVu Sans',
    'axes.titlepad':    14,        'axes.titlesize':    12,
    'axes.titleweight': 'bold',
})
CAT_COLORS = {
    'Food & Dining':'#89DCEB', 'Rent & Housing':'#F38BA8',
    'Transport':'#A6E3A1',     'Entertainment':'#CBA6F7',
    'Shopping':'#FAB387',      'Health':'#F9E2AF',
    'Education':'#74C7EC',     'Utilities':'#94E2D5',
}
PAY_COLORS = {
    'UPI':'#89DCEB', 'Cash':'#A6E3A1', 'Credit Card':'#F38BA8',
    'Debit Card':'#CBA6F7', 'Net Banking':'#FAB387'
}
COLORS = list(CAT_COLORS.values())
print("✅ Libraries loaded. FinTech dark theme configured.")

In [ ]:
# CELL 2 — Generate Synthetic Expense Dataset
# Creates ~700 realistic expense transactions for a full year (2024).
# Covers 8 categories, 5 payment modes, seasonal spend variation.
np.random.seed(42)
categories  = list(CAT_COLORS.keys())
pay_methods = ['UPI','Cash','Credit Card','Debit Card','Net Banking']
descriptions = {
    'Food & Dining':  ['Zomato Order','Swiggy','Restaurant Bill','Tea & Snacks','Grocery Store'],
    'Rent & Housing': ['Monthly Rent','Electricity Bill','Water Bill','Maintenance Charges'],
    'Transport':      ['Ola Cab','Auto Rickshaw','Petrol Fill','Metro Card','Bus Pass'],
    'Entertainment':  ['Netflix Subscription','Movie Ticket','Concert Ticket','Spotify','Gaming'],
    'Shopping':       ['Amazon Order','Flipkart Purchase','Clothes','Accessories','Electronics'],
    'Health':         ['Medicine Purchase','Doctor Consultation','Gym Membership','Lab Test'],
    'Education':      ['Online Course Fee','Books','Coaching Class','Stationery'],
    'Utilities':      ['Mobile Recharge','Internet Bill','DTH Recharge','Gas Bill'],
}
cat_spend_range = {
    'Food & Dining':(80,600),   'Rent & Housing':(100,2000),
    'Transport':(30,350),       'Entertainment':(100,800),
    'Shopping':(150,2500),      'Health':(80,1200),
    'Education':(200,1500),     'Utilities':(80,600),
}
cat_budget = {
    'Food & Dining':5500, 'Rent & Housing':12000, 'Transport':2800,
    'Entertainment':2200, 'Shopping':4000,         'Health':1800,
    'Education':3500,     'Utilities':2000,
}

rows = []
for date in pd.date_range('2024-01-01','2024-12-31',freq='D'):
    for _ in range(np.random.randint(1,4)):
        cat = np.random.choice(categories, p=[0.28,0.10,0.18,0.10,0.12,0.07,0.07,0.08])
        lo,hi = cat_spend_range[cat]
        rows.append({
            'date': date, 'category': cat,
            'description': np.random.choice(descriptions[cat]),
            'amount': round(np.random.uniform(lo,hi), 2),
            'payment_method': np.random.choice(pay_methods, p=[0.40,0.15,0.25,0.12,0.08]),
            'month': date.month, 'month_name': date.strftime('%b'),
            'day_of_week': date.strftime('%A'), 'week': date.isocalendar()[1],
        })

df = pd.DataFrame(rows)
df['quarter'] = pd.PeriodIndex(df['date'], freq='Q').astype(str)
df['quarter_short'] = df['quarter'].str.replace('2024Q','Q')
df.to_csv('expense_data.csv', index=False)

total_spend = df['amount'].sum()
monthly_avg = df.groupby('month')['amount'].sum().mean()
print(f"✅ Dataset created: {len(df):,} transactions")
print(f"💸 Total Annual Spend : ₹{total_spend:,.0f}")
print(f"📊 Monthly Avg Spend  : ₹{monthly_avg:,.0f}")
df.head(8)

In [ ]:
# CELL 3 — Data Overview & KPIs
# Quick stats check and key performance indicators.
print("="*55)
print("  EXPENSE TRACKER — DATA OVERVIEW")
print("="*55)
print(f"\n🔢 Total Transactions  : {len(df):,}")
print(f"📅 Date Range          : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"❌ Null Values          : {df.isnull().sum().sum()}")
print(f"🏷️  Categories          : {list(df['category'].unique())}")
print(f"💳 Payment Methods     : {list(df['payment_method'].unique())}")
print(f"\n💡 KEY METRICS:")
print(f"  🏆 Top Category      : {df.groupby('category')['amount'].sum().idxmax()}")
print(f"  💳 Top Payment Mode  : {df.groupby('payment_method')['amount'].sum().idxmax()}")
print(f"  📈 Highest Day       : ₹{df.groupby('date')['amount'].sum().max():,.0f}")
print(f"  📉 Lowest Day        : ₹{df.groupby('date')['amount'].sum().min():,.0f}")
print("\n📊 Summary Statistics:")
df[['amount']].describe().round(2)

In [ ]:
# CELL 4 — Main Analytics Dashboard (5 Charts in 1 Figure)
# Core EDA: monthly trend, category breakdown, payment modes, weekday patterns.
fig = plt.figure(figsize=(16, 11))
fig.patch.set_facecolor('#0E0E16')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.48, wspace=0.38)

# A — Monthly spend trend (line chart)
ax1 = fig.add_subplot(gs[0, :2]); ax1.set_facecolor('#14141F')
monthly = df.groupby('month')['amount'].sum()
mlbls   = [calendar.month_abbr[m] for m in monthly.index]
ax1.plot(mlbls, monthly.values, color='#89DCEB', linewidth=2.5, marker='o',
         markersize=7, markerfacecolor='#CBA6F7', markeredgecolor='white', markeredgewidth=1.2)
ax1.fill_between(mlbls, monthly.values, alpha=0.12, color='#89DCEB')
for i,(l,v) in enumerate(zip(mlbls, monthly.values)):
    ax1.text(i, v + monthly.values.max()*0.025, f'₹{v/1000:.0f}K', ha='center', fontsize=8, color='#CDD6F4')
ax1.set_title('💸  Monthly Spending Trend (Jan–Dec 2024)')
ax1.set_ylabel('Total Spend (₹)'); ax1.set_xlabel('Month')
ax1.grid(True, alpha=0.25)
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

# B — Category donut
ax2 = fig.add_subplot(gs[0, 2]); ax2.set_facecolor('#14141F')
cat_totals = df.groupby('category')['amount'].sum().sort_values(ascending=False)
wedges, texts, autotexts = ax2.pie(
    cat_totals.values, labels=cat_totals.index, autopct='%1.1f%%',
    colors=[CAT_COLORS[c] for c in cat_totals.index], startangle=90,
    wedgeprops=dict(width=0.58, edgecolor='#0E0E16', linewidth=1.5),
    textprops={'color':'#CDD6F4','fontsize':7.5}, pctdistance=0.78)
for at in autotexts: at.set_fontsize(7); at.set_color('#0E0E16'); at.set_fontweight('bold')
ax2.set_title('🥧  Spending by Category')

# C — Payment method horizontal bar
ax3 = fig.add_subplot(gs[1, 0]); ax3.set_facecolor('#14141F')
pay_totals = df.groupby('payment_method')['amount'].sum().sort_values(ascending=True)
ax3.barh(pay_totals.index, pay_totals.values,
         color=[PAY_COLORS.get(p,'#89DCEB') for p in pay_totals.index], alpha=0.87, edgecolor='none')
ax3.set_title('💳  Spend by Payment Method')
ax3.set_xlabel('Total Amount (₹)')
ax3.grid(True, axis='x', alpha=0.25)
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)
for bar, val in zip(ax3.patches, pay_totals.values):
    ax3.text(val+300, bar.get_y()+bar.get_height()/2, f'₹{val/1000:.0f}K', va='center', color='#CDD6F4', fontsize=8.5)

# D — Category totals horizontal bar
ax4 = fig.add_subplot(gs[1, 1]); ax4.set_facecolor('#14141F')
top_cats = cat_totals.sort_values(ascending=True)
ax4.barh(top_cats.index, top_cats.values,
         color=[CAT_COLORS[c] for c in top_cats.index], alpha=0.87, edgecolor='none')
ax4.set_title('📊  Total Spend by Category')
ax4.set_xlabel('Amount (₹)')
ax4.grid(True, axis='x', alpha=0.25)
ax4.spines['top'].set_visible(False); ax4.spines['right'].set_visible(False)
for bar, val in zip(ax4.patches, top_cats.values):
    ax4.text(val+200, bar.get_y()+bar.get_height()/2, f'₹{val/1000:.0f}K', va='center', color='#CDD6F4', fontsize=8.5)

# E — Weekday spend pattern
ax5 = fig.add_subplot(gs[1, 2]); ax5.set_facecolor('#14141F')
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
weekday_avg = df.groupby('day_of_week')['amount'].mean().reindex(day_order)
wknd_colors = ['#FAB387' if d in ['Saturday','Sunday'] else '#89DCEB' for d in day_order]
ax5.bar(weekday_avg.index, weekday_avg.values, color=wknd_colors, alpha=0.85, edgecolor='none')
ax5.set_title('📅  Avg Spend by Day of Week\n(Orange = Weekend)')
ax5.set_ylabel('Avg per Transaction (₹)')
ax5.tick_params(axis='x', rotation=35)
ax5.grid(True, axis='y', alpha=0.25)
ax5.spines['top'].set_visible(False); ax5.spines['right'].set_visible(False)

fig.suptitle('💰  Personal Expense Tracker — 2024 Analytics Dashboard', fontsize=15, color='#E6E8EF', y=1.01)
plt.tight_layout()
plt.savefig('chart1_main_dashboard.png', dpi=150, bbox_inches='tight', facecolor='#0E0E16')
plt.show()
print("💾 Saved: chart1_main_dashboard.png")

In [ ]:
# CELL 5 — Monthly Category Heatmap
# Shows every category's spend for every month — reveals seasonal patterns.
monthly_cat = df.pivot_table(values='amount', index='category', columns='month', aggfunc='sum')
monthly_cat.columns = [calendar.month_abbr[m] for m in monthly_cat.columns]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(monthly_cat/1000, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.4, linecolor='#0E0E16', ax=ax,
            cbar_kws={'label': 'Spend (₹ Thousands)'})
ax.set_title('🗓️  Monthly Spending Heatmap by Category (₹ in Thousands)', fontsize=13, pad=12)
ax.set_xlabel('Month'); ax.set_ylabel('')
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig('chart2_monthly_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0E0E16')
plt.show()
print("💾 Saved: chart2_monthly_heatmap.png")

In [ ]:
# CELL 6 — Feature Engineering
# Adds useful derived columns for deeper analysis.
df['is_weekend']    = df['day_of_week'].isin(['Saturday','Sunday']).astype(int)
df['is_upi']        = (df['payment_method'] == 'UPI').astype(int)
df['is_high_spend'] = (df['amount'] > df['amount'].quantile(0.90)).astype(int)
df['month_sin']     = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos']     = np.cos(2 * np.pi * df['month'] / 12)

print("✅ New features added:")
print("  is_weekend    → 1 if Saturday/Sunday")
print("  is_upi        → 1 if payment via UPI")
print("  is_high_spend → 1 if transaction in top 10% by amount")
print("  month_sin/cos → cyclical month encoding")
print(f"\n🔴 High-spend transactions: {df['is_high_spend'].sum()} ({df['is_high_spend'].mean()*100:.1f}%)")
print(f"📅 Weekend transactions   : {df['is_weekend'].sum()} ({df['is_weekend'].mean()*100:.1f}%)")
df[['date','category','amount','payment_method','is_weekend','is_high_spend']].head(6)

In [ ]:
# CELL 7 — Budget vs Actual Analysis
# Compares monthly budgets to actual spending per category.
monthly_cat_avg = df.groupby('category')['amount'].sum() / 12
budget_df = pd.DataFrame({'Budget': cat_budget, 'Actual': monthly_cat_avg}).round(0)
budget_df['Diff']   = budget_df['Actual'] - budget_df['Budget']
budget_df['Status'] = budget_df['Diff'].apply(lambda d: '🔴 OVER BUDGET' if d > 0 else '🟢 WITHIN BUDGET')
budget_df = budget_df.sort_values('Actual', ascending=False)

print("="*70)
print(f"  {'Category':<20} {'Budget':>10} {'Actual':>10} {'Difference':>12} {'Status'}")
print("-"*70)
for idx, row in budget_df.iterrows():
    sign = '+' if row['Diff'] > 0 else ''
    print(f"  {idx:<20} ₹{row['Budget']:>8,.0f} ₹{row['Actual']:>8,.0f}  {sign}₹{row['Diff']:>8,.0f}  {row['Status']}")
print("="*70)
n_over = (budget_df['Diff'] > 0).sum()
print(f"\n⚠️  {n_over} out of {len(budget_df)} categories over budget this month")

In [ ]:
# CELL 8 — Budget vs Actual Chart
# Visual comparison — most important chart for FinTech/DA interviews.
fig, ax = plt.subplots(figsize=(12, 6))
budget_sorted = budget_df.sort_values('Actual', ascending=True)
x = np.arange(len(budget_sorted)); w = 0.35

ax.barh(x - w/2, budget_sorted['Budget'], w, label='Budget',
        color='#A6E3A1', alpha=0.75, edgecolor='none')
bar_actual_colors = [('#F38BA8' if a > b else '#89DCEB')
                     for a, b in zip(budget_sorted['Actual'], budget_sorted['Budget'])]
ax.barh(x + w/2, budget_sorted['Actual'], w, label='Actual Spend',
        color=bar_actual_colors, alpha=0.85, edgecolor='none')

ax.set_yticks(x); ax.set_yticklabels(budget_sorted.index)
ax.set_title('🎯  Budget vs Actual Monthly Spend by Category\n(Red = Over Budget  |  Teal = Within Budget)', fontsize=12)
ax.set_xlabel('Amount (₹)')
ax.legend(framealpha=0.15, labelcolor='white', fontsize=10)
ax.grid(True, axis='x', alpha=0.25)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

for i, (idx, row) in enumerate(budget_sorted.iterrows()):
    if row['Diff'] > 0:
        ax.text(row['Actual'] + 50, i + w/2,
                f'+₹{row["Diff"]:.0f} ⚠️', va='center', color='#F38BA8', fontsize=8)

plt.tight_layout()
plt.savefig('chart3_budget_vs_actual.png', dpi=150, bbox_inches='tight', facecolor='#0E0E16')
plt.show()
print("💾 Saved: chart3_budget_vs_actual.png")

In [ ]:
# CELL 9 — Top 10 Transactions + Weekly Trend
# Identifies big-ticket spending moments and weekly patterns.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Top 10 single transactions
top10 = df.nlargest(10, 'amount')[['date','description','category','amount']].reset_index(drop=True)
labels_t10 = [f"{row['description']} ({row['date'].strftime('%d %b')})" for _, row in top10.iterrows()]
ax1.barh(range(10), top10['amount'].values[::-1],
         color=[CAT_COLORS[c] for c in top10['category'].values[::-1]], alpha=0.85, edgecolor='none')
ax1.set_yticks(range(10)); ax1.set_yticklabels(labels_t10[::-1], fontsize=9)
ax1.set_title('💎  Top 10 Highest Single Transactions')
ax1.set_xlabel('Amount (₹)')
ax1.grid(True, axis='x', alpha=0.25)
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)
for bar, val in zip(ax1.patches, top10['amount'].values[::-1]):
    ax1.text(val + 20, bar.get_y() + bar.get_height()/2, f'₹{val:,.0f}', va='center', color='#CDD6F4', fontsize=8.5)

# Weekly trend
weekly = df.groupby('week')['amount'].sum()
avg_w  = weekly.mean()
ax2.plot(weekly.index, weekly.values, color='#CBA6F7', linewidth=2, alpha=0.9, label='Weekly Spend')
ax2.fill_between(weekly.index, weekly.values, alpha=0.1, color='#CBA6F7')
ax2.axhline(y=avg_w, color='#FAB387', linestyle='--', linewidth=1.5, alpha=0.8,
            label=f'Avg: ₹{avg_w:,.0f}/wk')
ax2.set_title('📈  Weekly Spending Trend')
ax2.set_xlabel('Week Number'); ax2.set_ylabel('Total Spend (₹)')
ax2.legend(framealpha=0.15, labelcolor='white', fontsize=9)
ax2.grid(True, alpha=0.25)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('chart4_top10_weekly.png', dpi=150, bbox_inches='tight', facecolor='#0E0E16')
plt.show()
print("💾 Saved: chart4_top10_weekly.png")

In [ ]:
# CELL 10 — Quarterly Overview + Savings Rate
# Business-level quarterly summary and personal savings analysis.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Quarterly totals
quarterly = df.groupby('quarter_short')['amount'].sum()
q_colors  = ['#89DCEB','#A6E3A1','#FAB387','#F38BA8']
bars_q = ax1.bar(quarterly.index, quarterly.values, color=q_colors, alpha=0.85, edgecolor='none', width=0.5)
for b, v in zip(bars_q, quarterly.values):
    ax1.text(b.get_x()+b.get_width()/2, v+150, f'₹{v/1000:.0f}K',
             ha='center', color='#CDD6F4', fontsize=11, fontweight='bold')
ax1.set_title('📆  Quarterly Spending Overview')
ax1.set_xlabel('Quarter'); ax1.set_ylabel('Total Spend (₹)')
ax1.grid(True, axis='y', alpha=0.25)
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

# Savings rate
assumed_income  = 45000
monthly_spend_s = df.groupby('month')['amount'].sum()
savings_pct     = ((assumed_income - monthly_spend_s) / assumed_income * 100).round(1)
sav_colors      = ['#3FB950' if s > 0 else '#F38BA8' for s in savings_pct]
ax2.bar([calendar.month_abbr[m] for m in savings_pct.index], savings_pct.values,
        color=sav_colors, alpha=0.85, edgecolor='none')
ax2.axhline(y=0, color='#6C7086', linewidth=1, alpha=0.7)
ax2.axhline(y=20, color='#A6E3A1', linewidth=1, linestyle=':', alpha=0.6, label='20% target')
ax2.set_title(f'💰  Monthly Savings Rate\n(Assumed Income: ₹{assumed_income:,}/mo)')
ax2.set_ylabel('Savings %')
ax2.tick_params(axis='x', rotation=35)
ax2.legend(framealpha=0.15, labelcolor='white', fontsize=9)
ax2.grid(True, axis='y', alpha=0.25)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('chart5_quarterly_savings.png', dpi=150, bbox_inches='tight', facecolor='#0E0E16')
plt.show()
print("💾 Saved: chart5_quarterly_savings.png")

In [ ]:
# CELL 11 — Spending Insights (Text Analysis)
# Auto-generates human-readable financial insights from the data.
monthly_spend_s  = df.groupby('month')['amount'].sum()
assumed_income   = 45000
savings_pct      = ((assumed_income - monthly_spend_s) / assumed_income * 100).round(1)
cat_monthly_avg  = df.groupby('category')['amount'].sum() / 12
over_budget_cats = [cat for cat in cat_budget if cat_monthly_avg[cat] > cat_budget[cat]]
best_month       = monthly_spend_s.idxmin()
worst_month      = monthly_spend_s.idxmax()
best_savings     = savings_pct.idxmax()
upi_share        = df[df['payment_method']=='UPI']['amount'].sum() / df['amount'].sum() * 100
weekend_avg      = df[df['is_weekend']==1]['amount'].mean()
weekday_avg_val  = df[df['is_weekend']==0]['amount'].mean()

print("💡 AUTO-GENERATED FINANCIAL INSIGHTS")
print("="*55)
print(f"\n📅 MONTHLY PATTERNS:")
print(f"  • Lowest spend month : {calendar.month_name[best_month]} (₹{monthly_spend_s[best_month]:,.0f})")
print(f"  • Highest spend month: {calendar.month_name[worst_month]} (₹{monthly_spend_s[worst_month]:,.0f})")
print(f"  • Best savings month : {calendar.month_name[best_savings]} ({savings_pct[best_savings]:.1f}% saved)")
print(f"\n💳 PAYMENT BEHAVIOR:")
print(f"  • UPI dominates at {upi_share:.1f}% of total spend value")
print(f"  • Top payment: {df.groupby('payment_method').size().idxmax()} by transaction count")
print(f"\n📅 WEEKEND vs WEEKDAY:")
print(f"  • Avg weekend transaction : ₹{weekend_avg:,.0f}")
print(f"  • Avg weekday transaction : ₹{weekday_avg_val:,.0f}")
pct_diff = (weekend_avg - weekday_avg_val) / weekday_avg_val * 100
print(f"  • Weekends cost {abs(pct_diff):.1f}% {'more' if pct_diff>0 else 'less'} per transaction")
print(f"\n⚠️  BUDGET ALERTS:")
if over_budget_cats:
    for cat in over_budget_cats:
        excess = cat_monthly_avg[cat] - cat_budget[cat]
        print(f"  🔴 {cat}: ₹{excess:,.0f} over monthly budget")
else:
    print("  🟢 All categories within budget!")
print(f"\n💹 SAVINGS SUMMARY:")
print(f"  • Avg monthly savings rate : {savings_pct.mean():.1f}%")
months_saved = (savings_pct > 0).sum()
print(f"  • Months with positive savings: {months_saved}/12")

In [ ]:
# CELL 12 — Final Report Summary
# Complete boxed summary — screenshot this for your GitHub README.
monthly_spend_s  = df.groupby('month')['amount'].sum()
assumed_income   = 45000
savings_pct      = ((assumed_income - monthly_spend_s) / assumed_income * 100).round(1)
cat_monthly_avg  = df.groupby('category')['amount'].sum() / 12
over_cats        = [c for c in cat_budget if cat_monthly_avg[c] > cat_budget[c]]
best_m           = monthly_spend_s.idxmin()
worst_m          = monthly_spend_s.idxmax()

print()
print("╔══════════════════════════════════════════════════════╗")
print("║       EXPENSE TRACKER APP — FINAL REPORT            ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  📅 Period            : Jan 2024 – Dec 2024         ║")
print(f"║  🧾 Total Transactions: {len(df):<27}║")
print(f"║  💸 Total Spend       : ₹{df['amount'].sum():>21,.0f}  ║")
print(f"║  📊 Monthly Avg       : ₹{df.groupby('month')['amount'].sum().mean():>21,.0f}  ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  🏆 Top Category      : {df.groupby('category')['amount'].sum().idxmax():<28}║")
print(f"║  💳 Top Payment Mode  : {df.groupby('payment_method')['amount'].sum().idxmax():<28}║")
print(f"║  📈 Highest Spend Mo  : {calendar.month_name[worst_m]:<28}║")
print(f"║  📉 Lowest Spend Mo   : {calendar.month_name[best_m]:<28}║")
print("╠══════════════════════════════════════════════════════╣")
over_str = ', '.join(over_cats[:3]) if over_cats else 'None — Under budget!'
print(f"║  ⚠️  Over-Budget Cats  : {over_str[:28]:<28}║")
print(f"║  💹 Avg Savings Rate  : {savings_pct.mean():.1f}%{'':<23}║")
print("╠══════════════════════════════════════════════════════╣")
print("║  📁 Output Files:                                    ║")
print("║     expense_data.csv                                ║")
print("║     chart1_main_dashboard.png                       ║")
print("║     chart2_monthly_heatmap.png                      ║")
print("║     chart3_budget_vs_actual.png                     ║")
print("║     chart4_top10_weekly.png                         ║")
print("║     chart5_quarterly_savings.png                    ║")
print("╚══════════════════════════════════════════════════════╝")